# Tafsir Lab — build the search index

Runs on Colab's **free T4**. About 20–40 minutes for the full corpus.

**This is not fine-tuning, and that is deliberate.** Nothing about the model
changes here. The model reads each passage once and records where it sits in
meaning-space, so a question can later be placed in the same space and the
nearest *real* passages found.

A fine-tuned model would learn to **write like** al-Ṭabarī — which is exactly
how you end up with fluent, confident commentary attributed to a scholar who
never wrote it. This pipeline never generates a word of tafsīr. It only points
at text a human wrote, which is why quoting is possible and inventing is not.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

## 1. Install

In [ ]:
!pip -q install sentence-transformers==3.* psycopg2-binary
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > T4 GPU')

## 2. Get the script

Either upload `embed_corpus.py` from `ml/` using the file pane on the left, or
paste it into a cell. Nothing else from the repo is needed.

In [ ]:
import os
assert os.path.exists('embed_corpus.py'), 'Upload ml/embed_corpus.py first (file pane, left)'
print('ready')

## 3. Connect

Paste the **pooler** connection string from Supabase → Project Settings →
Database → Connection string → *Transaction pooler*.

It must be the pooler host. The direct `db.<ref>.supabase.co` host is IPv6-only
and Colab will simply hang on it.

`getpass` keeps it out of the notebook's saved output, so the file stays safe
to share.

In [ ]:
import getpass, os
os.environ['DATABASE_URL'] = getpass.getpass('DATABASE_URL (pooler): ')

import psycopg2
with psycopg2.connect(os.environ['DATABASE_URL'], connect_timeout=30) as c:
    with c.cursor() as cur:
        cur.execute('select count(*) from "TafsirEntry"')
        entries = cur.fetchone()[0]
        cur.execute('select count(*) from "TafsirChunk"')
        chunks = cur.fetchone()[0]
print(f'{entries:,} tafsir entries, {chunks:,} already embedded')

## 4. A small run first

Embed a few thousand entries and look at the results before committing the GPU
session to the whole corpus. If retrieval is poor, it is far cheaper to find
out now.

In [ ]:
# Smoke test on the smallest edition before committing to the full run.
!python embed_corpus.py --source ar-tafsir-al-mukhtasar --limit 200


## 5. Check that retrieval actually works

Ask something and look at what comes back. If the passages are not about what
you asked, stop here — a bigger index will not fix a bad one.

In [ ]:
from sentence_transformers import SentenceTransformer
import psycopg2, os

m = SentenceTransformer('intfloat/multilingual-e5-small')

def search(question, k=5, source=None):
    # 'query: ' is required by e5 and must mirror the 'passage: ' prefix used
    # when embedding. Mismatched prefixes degrade results quietly.
    v = m.encode('query: ' + question, normalize_embeddings=True)
    lit = '[' + ','.join(f'{x:.6f}' for x in v) + ']'

    # TafsirChunk stores character offsets, not a copy of the text, so the
    # snippet has to be sliced back out of TafsirEntry. Storing the text twice
    # would have doubled the largest table in the database.
    sql = '''select s.name, ch."verseKey",
                    substr(e.content, ch."startChar" + 1,
                           least(240, ch."endChar" - ch."startChar")) as snippet,
                    1 - (ch.embedding <=> %s::halfvec) as score
             from "TafsirChunk" ch
             join "TafsirSource" s on s.id = ch."sourceId"
             join "TafsirEntry"  e on e."sourceId" = ch."sourceId"
                                  and e."verseKey" = ch."verseKey"
             where ch.embedding is not null {flt}
             order by ch.embedding <=> %s::halfvec limit %s'''
    args = [lit]
    flt = ''
    if source:
        flt = 'and s.slug = %s'
        args.append(source)
    args += [lit, k]
    with psycopg2.connect(os.environ['DATABASE_URL']) as c:
        with c.cursor() as cur:
            cur.execute(sql.format(flt=flt), args)
            rows = cur.fetchall()
    if not rows:
        print('No embedded chunks yet — run the smoke test (cell 8) first.')
    for name, vk, snip, score in rows:
        print(f'\n[{score:.3f}] {name} — {vk}\n{snip}…')

search('What does it say about patience in hardship?')


## 6. The full run

Resumable: every entry already embedded is skipped, so if Colab disconnects,
re-run this cell and it continues from where it stopped.

In [ ]:
# The editions that fit the 500 MB free tier, smallest first so the cheap wins
# land before anything can time out.
#
# Deliberately NOT every source. Embedding all 113 needs ~853 MB, and passing
# the quota does not merely stop the embedding — it stops writes for your
# users. The real omissions are al-Qurtubi and al-Razi; Ibn Kathir Arabic is
# skipped only because the English edition of the same work is already listed.
EDITIONS = [
    "ar-tafsir-al-mukhtasar",
    "ar-tafsir-al-jalalayn",
    "en-tafsir-al-mukhtasar",
    "tafsir-al-jalalayn",
    "tafsir-al-baydawi",
    "ar-tafsir-as-saadi",
    "ar-tafsir-al-tabari",
    "en-tafisr-ibn-kathir",
]

import subprocess, time
start = time.time()
for i, slug in enumerate(EDITIONS, 1):
    print("\n" + "=" * 64)
    print("[%d/%d] %s" % (i, len(EDITIONS), slug))
    print("=" * 64, flush=True)
    t = time.time()
    # check=False: one edition failing must not discard the hours already
    # committed by the editions before it. The script is resumable, so a
    # failure can be re-run on its own afterwards.
    subprocess.run(["python", "embed_corpus.py", "--source", slug], check=False)
    print("  -> %.1f min" % ((time.time() - t) / 60), flush=True)
print("\nall editions done in %.0f min" % ((time.time() - start) / 60))


## 7. Index — not needed on the free tier


In [ ]:
# DO NOT RUN unless the database has room to spare.
#
# An HNSW index makes vector search sub-millisecond, and at this scale that
# buys nothing worth having: about 75,000 vectors is an exact scan of roughly
# 100 ms, and the app narrows the candidate set before comparing anyway. The
# index would cost ~1.4x the vectors themselves — around 77 MB — on a 500 MB
# free tier that will already be at ~88% once embedding finishes.
#
# Build it only if you move to a paid plan. Until then this cell just reports
# where the database stands.
import psycopg2, os

with psycopg2.connect(os.environ['DATABASE_URL']) as c:
    with c.cursor() as cur:
        cur.execute("select pg_size_pretty(pg_database_size(current_database()))")
        size = cur.fetchone()[0]
        cur.execute('select count(*), count(embedding) from "TafsirChunk"')
        n, emb = cur.fetchone()

print(f'database:  {size}  of 500 MB free tier')
print(f'chunks:    {n:,}  ({emb:,} embedded)')
print()
print('No index built — exact scan is fast enough at this size, and the index')
print('would cost ~77 MB the free tier does not have.')
